# Model 1 observation experiments on Kaggle

Select **GPU T4 x2**, enable Internet, and attach the G1 mesh dataset. Run the cells from the top.

The default `experiment = "upstream"` compares the 103-input Playground G1 observation with the 67-input M1 control. Attach your saved `m1_160k_control` folder to reuse it. If the folder is unavailable, the notebook trains the control in this session as well. Each run uses 160,000 steps, eight environments (four per T4), and seed 42.

Set `experiment = "velocity"` to compare 67 and 69 inputs or `experiment = "baseline"` for the original 1.6M-step workflow.

In [ ]:
import json
import os
from pathlib import Path
import shutil
import statistics
import subprocess
import sys
from tempfile import gettempdir
from zipfile import ZipFile

repo = Path("/kaggle/working/Bipedal-Locomotion")
inputs = Path("/kaggle/input")
branch = "m3-kaggle"
url = "https://github.com/TotallyNotMinh/Bipedal-Locomotion.git"

if not repo.exists():
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", url, str(repo)], check=True)
active_branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=repo, text=True).strip()
if active_branch != branch:
    raise RuntimeError(f"Expected {branch} in {repo}, found {active_branch!r}. Check the existing checkout.")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=repo, check=True)

experiment = "upstream"  # Also accepts "velocity" or "baseline".
control_reference = None  # Optionally set Path("/kaggle/input/.../m1_160k_control").

if experiment in ("upstream", "velocity"):
    run = repo / "results/m1_160k_control"
    if experiment == "upstream":
        if control_reference is None and not (run / "training.json").is_file():
            matches = sorted(inputs.rglob("m1_160k_control/training.json"))
            if len(matches) > 1:
                raise ValueError(f"Set control_reference to one of: {[path.parent for path in matches]}")
            if matches:
                control_reference = matches[0].parent
        if control_reference is not None:
            run = control_reference
            if not (run / "training.json").is_file():
                raise FileNotFoundError(f"Missing completed control at {run}")
            print(f"Using saved control from {run}")
        variant_run = repo / "results/m1_160k_upstream_obs"
    else:
        variant_run = repo / "results/m1_160k_velocity_obs"
    budget = 160_000
elif experiment == "baseline":
    run = repo / "results/m1_1p6m_env8_lr1e4"
    variant_run = None
    budget = 1_600_000
else:
    raise ValueError("Choose upstream, velocity, or baseline.")
needs_training = any(not (folder / "training.json").is_file() for folder in (run, variant_run) if folder is not None)
num_envs = 8
seed = 42
env = {**os.environ, "XLA_PYTHON_CLIENT_PREALLOCATE": "false"}

def read(folder, filename):
    return json.loads((folder / filename).read_text())

## Load the G1 meshes for training

For training, the notebook copies missing STL files from `/kaggle/input/datasets/trung1311/robot-assets/results/m3_robot_assets` into the trainer’s mesh directory. Checkpoint verification of a completed M1 run can proceed without the meshes.

In [ ]:
meshes = repo / "results/m3_robot_assets"
expected_meshes = set("""head_link.STL
left_ankle_pitch_link.STL
left_ankle_roll_link.STL
left_elbow_link.STL
left_hip_pitch_link.STL
left_hip_roll_link.STL
left_hip_yaw_link.STL
left_knee_link.STL
left_rubber_hand.STL
left_shoulder_pitch_link.STL
left_shoulder_roll_link.STL
left_shoulder_yaw_link.STL
left_wrist_pitch_link.STL
left_wrist_roll_link.STL
left_wrist_yaw_link.STL
logo_link.STL
pelvis.STL
pelvis_contour_link.STL
right_ankle_pitch_link.STL
right_ankle_roll_link.STL
right_elbow_link.STL
right_hip_pitch_link.STL
right_hip_roll_link.STL
right_hip_yaw_link.STL
right_knee_link.STL
right_rubber_hand.STL
right_shoulder_pitch_link.STL
right_shoulder_roll_link.STL
right_shoulder_yaw_link.STL
right_wrist_pitch_link.STL
right_wrist_roll_link.STL
right_wrist_yaw_link.STL
torso_link_rev_1_0.STL
waist_roll_link_rev_1_0.STL
waist_yaw_link_rev_1_0.STL""".split())
missing = expected_meshes.difference(p.name for p in meshes.glob("*.STL") if p.stat().st_size)
if missing and needs_training:
    meshes.mkdir(parents=True, exist_ok=True)
    source_dir = inputs / "datasets/trung1311/robot-assets/results/m3_robot_assets"
    archives = [] if source_dir.is_dir() else sorted(inputs.rglob("m3-robot-assets.zip"))
    if archives:
        with ZipFile(archives[0]) as bundle:
            for name in sorted(missing):
                with bundle.open(f"results/m3_robot_assets/{name}") as source:
                    (meshes / name).write_bytes(source.read())
    else:
        if not source_dir.is_dir():
            folders = [p for p in inputs.rglob("m3_robot_assets") if p.is_dir()
                       and all((p / name).is_file() for name in missing)]
            if not folders:
                raise FileNotFoundError(f"Attach the robot-assets dataset at {source_dir} or attach m3-robot-assets.zip.")
            source_dir = folders[0]
        for name in sorted(missing):
            shutil.copyfile(source_dir / name, meshes / name)
    missing = {name for name in expected_meshes if not (meshes / name).stat().st_size}
    if missing:
        raise ValueError(f"Empty G1 mesh files: {sorted(missing)}")

## Enable the selected experiment

The cells add trainer support to older checkouts as needed. A conflicting edit stops the update before training starts.

In [ ]:
patch = r'''diff --git a/scripts/train_kaggle.py b/scripts/train_kaggle.py
index 17d1537..7619bb2 100644
--- a/scripts/train_kaggle.py
+++ b/scripts/train_kaggle.py
@@ -31,33 +31,40 @@ def main():
                         help="Path to terrain configuration YAML")
     parser.add_argument("--num-envs", type=int, default=None,
                         help="Total parallel environments (sharded across GPUs)")
-    parser.add_argument("--history-len", type=int, default=5,
-                        help="Number of history timesteps (k)")
+    parser.add_argument("--history-len", type=int,
+                        help="Number of history timesteps (M1: 0, other models default to 5)")
     parser.add_argument("--seed", type=int, default=42,
                         help="Master PRNG seed")
     parser.add_argument("--output-dir", type=str, default=None,
                         help="Checkpoint and logs output directory")
-    parser.add_argument("--num-timesteps", type=int, help="Exact M3 environment step budget")
-    parser.add_argument("--smoke", action="store_true", help="Run two M3 optimizer updates (pilot default: ten)")
+    parser.add_argument("--num-timesteps", type=int, help="Exact M1 or M3 environment step budget")
+    parser.add_argument("--smoke", action="store_true", help="Run two PPO optimizer updates (pilot default: ten)")
     args = parser.parse_args()
-    if args.model == "M3":
+    if args.history_len is None:
+        args.history_len = 0 if args.model == "M1" else 5
+    if args.model in ("M1", "M3"):
+        if args.model == "M1" and args.history_len != 0:
+            parser.error("M1 uses only the current observation (--history-len 0)")
+        if args.model == "M3" and args.history_len < 1:
+            parser.error("M3 history length must be positive")
         if args.category is not None or args.difficulty != 0 or Path(args.terrain_config).resolve() != ROOT / "configs/terrain.yaml":
-            parser.error("M3 uses the established terrain configuration and curriculum")
-        if args.history_len < 1 or any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
-            parser.error("History length, environment count, and step budget must be positive")
+            parser.error(f"{args.model} uses the established terrain configuration and curriculum")
+        if any(v is not None and v <= 0 for v in (args.num_envs, args.num_timesteps)):
+            parser.error("Environment count and step budget must be positive")
         if args.smoke and args.num_timesteps is not None:
             parser.error("Choose either --smoke or an explicit --num-timesteps")
-        args.output_dir = args.output_dir or ("results/m3_smoke" if args.smoke else "results/m3_pilot")
+        kind = "smoke" if args.smoke else "pilot"
+        args.output_dir = args.output_dir or f"results/{args.model.lower()}_{kind}"
         try:
             from src.training.m3_pilot import run
         except ImportError as exc:
-            parser.exit(2, f"M3 needs the configured Playground/Brax runtime: {exc}\n")
+            parser.exit(2, f"{args.model} needs the configured Playground/Brax runtime: {exc}\n")
         run(args)
         # Release the trainer's GPU allocations before verifying in a fresh interpreter.
         os.execv(sys.executable, [sys.executable, "-m", "src.training.m3_pilot", args.output_dir])
 
     if args.num_timesteps is not None or args.smoke:
-        parser.error("--num-timesteps and --smoke apply to M3")
+        parser.error("--num-timesteps and --smoke apply to M1 and M3")
     os.environ.update(XLA_PYTHON_CLIENT_PREALLOCATE="false",
                       XLA_PYTHON_CLIENT_MEM_FRACTION="0.85", TF_FORCE_GPU_ALLOW_GROWTH="true")
     args.num_envs = args.num_envs or 4096
diff --git a/src/training/m3_history.py b/src/training/m3_history.py
index 03ad757..5120aab 100644
--- a/src/training/m3_history.py
+++ b/src/training/m3_history.py
@@ -2,8 +2,10 @@
 
 
 def initial_input(obs, action_width, length, xp):
-    if length < 1:
-        raise ValueError("history length must be positive")
+    if length == 0:
+        return obs
+    if length < 0:
+        raise ValueError("history length must be nonnegative")
     shape = (*obs.shape[:-1], length * (obs.shape[-1] + action_width))
     return xp.concatenate((obs, xp.zeros(shape, dtype=obs.dtype)), axis=-1)
 
@@ -11,6 +13,8 @@ def initial_input(obs, action_width, length, xp):
 def next_input(previous, action, observation, done, xp):
     """Advance one transition. On autoreset, observation is the new reset observation."""
     width = observation.shape[-1]
+    if previous.shape[-1] == width:
+        return observation
     pair_width = width + action.shape[-1]
     history = xp.concatenate(
         (previous[..., width + pair_width:], previous[..., :width], action), axis=-1
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 256e9f4..0881c33 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -1,4 +1,4 @@
-"""Bounded M3 PPO training with matched initial/final evaluations."""
+"""Bounded M1 and M3 PPO training with matched initial/final evaluations."""
 
 from functools import partial
 from hashlib import sha256
@@ -65,7 +65,7 @@ def evaluate(env, policy, horizon):
             values = np.asarray([state.reward, state.metrics["m3_distance"],
                                  state.metrics["m3_tracking"], state.done])
             if not np.isfinite(values).all():
-                raise ValueError("Non-finite M3 evaluation transition")
+                raise ValueError("Non-finite evaluation transition")
             reward += float(values[0])
             distance += float(values[1])
             error += float(values[2])
@@ -177,7 +177,7 @@ def run(args):
     width = actor_shape.shape[-1]
     expected = width + args.history_len * (width + env.action_size)
     if observation_shapes["state"] != (expected,):
-        raise ValueError("Simulator observation shape disagrees with the M3 history contract")
+        raise ValueError("Simulator observation shape disagrees with the configured actor width")
 
     ppo = locomotion_params.brax_ppo_config(TASK).to_dict()
     ppo["learning_rate"] = 0.0001
@@ -189,7 +189,8 @@ def run(args):
     ppo.update(num_envs=nenv, batch_size=nenv, num_minibatches=1, num_updates_per_batch=1,
                num_timesteps=budget, num_evals=1, num_eval_envs=nenv, num_resets_per_eval=0,
                run_evals=False)
-    settings = dict(model="M3", architecture="PPO MLP with observation–action history stacking",
+    settings = dict(model=args.model, architecture=("PPO MLP with current observation" if args.model == "M1"
+                                                 else "PPO MLP with observation–action history stacking"),
         task=TASK, seed=args.seed, actor_fields=[
         "pelvis gyro", "projected gravity", "velocity command", "joint offsets", "joint velocities"],
         observation_width=width, actions=env.action_size, actor_width=expected,
@@ -201,7 +202,7 @@ def run(args):
         observation_shapes=observation_shapes, optimizer_updates=budget // unit,
         terrain_config=str(terrain_path.resolve()), output_dir=str(out),
         devices=[f"{d}: {d.device_kind}" for d in jax.local_devices()])
-    print("Resolved M3 run configuration", flush=True)
+    print(f"Resolved {args.model} run configuration", flush=True)
     print(json.dumps(settings, indent=2, allow_nan=False), flush=True)
     save_json(out / "settings.json", settings)
 
'''
def check_patch(*flags):
    return subprocess.run(["git", "apply", *flags, "-"], cwd=repo, input=patch,
                          text=True, capture_output=True)

# A newer checkout can support M1 even when the old patch no longer reverses.
m1_check = subprocess.run([sys.executable, "scripts/train_kaggle.py", "--model", "M1", "--history-len", "1"],
                          cwd=repo, text=True, capture_output=True)
supports_m1 = m1_check.returncode == 2 and "M1 uses only the current observation" in m1_check.stderr
if not supports_m1 and check_patch("--reverse", "--check").returncode != 0:
    result = check_patch("--check")
    if result.returncode:
        raise RuntimeError("Use the updated m3-kaggle checkout. " + result.stderr)
    check_patch().check_returncode()
print("M1 trainer ready.")

In [ ]:
if variant_run is not None:
    help_text = subprocess.check_output([sys.executable, "scripts/train_kaggle.py", "--help"], cwd=repo, text=True)
    if "--actor-velocity" not in help_text or "--fixed-command-eval" not in help_text:
        velocity_patch = r'''diff --git a/scripts/render_m3_replay.py b/scripts/render_m3_replay.py
index 5985f80..1ce6047 100644
--- a/scripts/render_m3_replay.py
+++ b/scripts/render_m3_replay.py
@@ -121,7 +121,8 @@ def main():
         local_scene = Path(temp) / "g1.xml"
         scene.write(local_scene)
         env = HistoryJoystick(
-            local_scene, config, settings["history_length"], Path("results/m3_robot_assets")
+            local_scene, config, settings["history_length"], Path("results/m3_robot_assets"),
+            actor_velocity=settings.get("actor_velocity", False),
         )
         reset, step = jax.jit(env.reset), jax.jit(env.step)
         trajectories = [
diff --git a/scripts/train_kaggle.py b/scripts/train_kaggle.py
index e9d79a4..a8cc596 100644
--- a/scripts/train_kaggle.py
+++ b/scripts/train_kaggle.py
@@ -41,9 +41,17 @@ def main():
     parser.add_argument("--smoke", action="store_true", help="Run two PPO optimizer updates (pilot default: ten)")
     parser.add_argument("--diagnose-ppo", action="store_true",
                         help="Record M1 KL and action saturation at each PPO update")
+    parser.add_argument("--actor-velocity", action="store_true",
+                        help="Append pelvis-frame forward and lateral velocity to the actor observation")
+    parser.add_argument("--fixed-command-eval", action="store_true",
+                        help="Evaluate the final M1 checkpoint at 0 and 0.5 m/s forward")
     args = parser.parse_args()
     if args.diagnose_ppo and args.model != "M1":
         parser.error("--diagnose-ppo applies to M1")
+    if args.actor_velocity and args.model not in ("M1", "M3"):
+        parser.error("--actor-velocity applies to M1 and M3")
+    if args.fixed_command_eval and args.model != "M1":
+        parser.error("--fixed-command-eval applies to M1")
     if args.history_len is None:
         args.history_len = 0 if args.model == "M1" else 5
     if args.model in ("M1", "M3"):
diff --git a/src/training/m3_env.py b/src/training/m3_env.py
index ae435b0..4c44505 100644
--- a/src/training/m3_env.py
+++ b/src/training/m3_env.py
@@ -9,7 +9,7 @@ from src.training.m3_history import initial_input, next_input
 
 
 class HistoryJoystick(joystick.Joystick):
-    def __init__(self, scene, config, length, mesh_directory):
+    def __init__(self, scene, config, length, mesh_directory, *, actor_velocity=False):
         mjx_env.MjxEnv.__init__(self, config)
         self._xml_path = str(scene)
         self._model_assets = base.get_assets()
@@ -23,16 +23,20 @@ class HistoryJoystick(joystick.Joystick):
         self._mjx_model = mjx_env.put_model(self._mj_model, impl=config.impl)
         self._post_init()
         self.history_length = length
+        self.actor_velocity = actor_velocity
 
     def _actor_obs(self, state):
         data = state.data
-        return jp.concatenate((
+        fields = (
             self.get_gyro(data, "pelvis"),
             -data.site_xmat[self._pelvis_imu_site_id][2],
             state.info["command"],
             data.qpos[7:] - self._default_pose,
             data.qvel[6:],
-        ))
+        )
+        if self.actor_velocity:
+            fields += (self.get_local_linvel(data, "pelvis")[:2],)
+        return jp.concatenate(fields)
 
     def reset(self, rng):
         state = super().reset(rng)
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 6b95506..74d6a81 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -30,6 +30,7 @@ from src.training.m3_scene import write_scene
 TASK = "G1JoystickFlatTerrain"
 EVAL_SEEDS = tuple(range(10_000, 10_010))
 ACTION_LIMIT_THRESHOLD = 0.95
+FIXED_FORWARD_COMMANDS = (0.0, 0.5)
 
 
 def enable_brax_jax_compatibility():
@@ -62,15 +63,31 @@ def with_action_metrics(compute_loss):
     return loss
 
 
-def evaluate(env, policy, horizon):
+def evaluate(env, policy, horizon, *, forward_command=None):
     reset = jax.jit(env.reset)
     step = jax.jit(env.step)
     act = jax.jit(policy)
+    if forward_command is not None:
+        command = jp.array([forward_command, 0., 0.], dtype=jp.float32)
+        contact_sensor = env.mj_model.sensor_adr[env._right_foot_left_foot_found_sensor]
+
+        @jax.jit
+        def with_command(state):
+            state = state.replace(info=dict(state.info, command=command))
+            return state.replace(obs=dict(state.obs, state=env._actor_obs(state)))
+
+        @jax.jit
+        def forward_velocity(data):
+            return env.get_local_linvel(data, "pelvis")[0]
+
     results = []
     for seed in EVAL_SEEDS:
         key = jax.random.PRNGKey(seed)
         state = reset(key)
         reward = distance = error = 0.
+        if forward_command is not None:
+            state = with_command(state)
+        forward_speed = 0.
         for length in range(1, horizon + 1):
             key, subkey = jax.random.split(key)
             action, _ = act(state.obs, subkey)
@@ -82,10 +99,18 @@ def evaluate(env, policy, horizon):
             reward += float(values[0])
             distance += float(values[1])
             error += float(values[2])
+            if forward_command is not None:
+                forward_speed += float(forward_velocity(state.data))
             if values[3]:
                 break
-        results.append(dict(seed=seed, episode_return=reward, distance_m=distance,
-                            fall=bool(values[3]), steps=length, tracking_error_mps=error / length))
+            if forward_command is not None:
+                state = with_command(state)
+        episode = dict(seed=seed, episode_return=reward, distance_m=distance,
+                       fall=bool(values[3]), steps=length, tracking_error_mps=error / length)
+        if forward_command is not None:
+            episode.update(mean_forward_mps=forward_speed / length,
+                           foot_to_foot_contact=bool(state.data.sensordata[contact_sensor] > 0))
+        results.append(episode)
     return dict(episodes=results, means={k: float(np.mean([r[k] for r in results]))
                                         for k in ("episode_return", "distance_m", "fall", "tracking_error_mps")})
 
@@ -184,7 +209,8 @@ def run(args):
     env_cfg.naconmax = min(env_cfg.naconmax, 256 * (nenv // devices))
     template = g1_constants.task_to_xml("flat_terrain")
     scene = write_scene(str(template), arena, out / "terrain")
-    env = HistoryJoystick(scene, env_cfg, args.history_len, Path("results/m3_robot_assets"))
+    env = HistoryJoystick(scene, env_cfg, args.history_len, Path("results/m3_robot_assets"),
+                          actor_velocity=args.actor_velocity)
     observation_shapes = env.observation_size
     actor_shape = jax.eval_shape(lambda key: env._actor_obs(env.reset(key)), jax.random.PRNGKey(0))
     width = actor_shape.shape[-1]
@@ -224,6 +250,9 @@ def run(args):
             action_limit_threshold=ACTION_LIMIT_THRESHOLD,
             kl="Rollout distribution vs. pre-SGD distribution with updated normalization",
         )
+    if args.actor_velocity:
+        settings["actor_velocity"] = True
+        settings["actor_fields"].append("pelvis forward and lateral velocity")
     print(f"Resolved {args.model} run configuration", flush=True)
     print(json.dumps(settings, indent=2, allow_nan=False), flush=True)
     save_json(out / "settings.json", settings)
@@ -253,6 +282,12 @@ def run(args):
             np.savez(out / "reload_probe.npz", expected_action=action, **sample.obs)
             label = "trained"
         save_json(out / f"{label}_evaluation.json", evaluate(env, policy, ppo["episode_length"]))
+        if label == "trained" and args.fixed_command_eval:
+            scenarios = [
+                dict(forward_command_mps=speed, **evaluate(env, policy, ppo["episode_length"], forward_command=speed))
+                for speed in FIXED_FORWARD_COMMANDS
+            ]
+            save_json(out / "fixed_command_evaluation.json", {"scenarios": scenarios})
 
     diagnostic_step = 0
 
'''
        applied = subprocess.run(["git", "apply", "--check", "-"], cwd=repo,
                                 input=velocity_patch, text=True, capture_output=True)
        if applied.returncode:
            raise RuntimeError("Use an updated m3-kaggle checkout with the velocity experiment. " + applied.stderr)
        subprocess.run(["git", "apply", "-"], cwd=repo, input=velocity_patch, text=True, check=True)
    print("M1 comparison trainer ready.")

In [ ]:
if experiment == "upstream":
    help_text = subprocess.check_output([sys.executable, "scripts/train_kaggle.py", "--help"], cwd=repo, text=True)
    if "--actor-upstream" not in help_text:
        upstream_patch = r'''diff --git a/scripts/render_m3_replay.py b/scripts/render_m3_replay.py
index 1ce6047..11bde93 100644
--- a/scripts/render_m3_replay.py
+++ b/scripts/render_m3_replay.py
@@ -123,6 +123,7 @@ def main():
         env = HistoryJoystick(
             local_scene, config, settings["history_length"], Path("results/m3_robot_assets"),
             actor_velocity=settings.get("actor_velocity", False),
+            actor_upstream=settings.get("actor_upstream", False),
         )
         reset, step = jax.jit(env.reset), jax.jit(env.step)
         trajectories = [
diff --git a/scripts/train_kaggle.py b/scripts/train_kaggle.py
index a8cc596..ab6b577 100644
--- a/scripts/train_kaggle.py
+++ b/scripts/train_kaggle.py
@@ -40,18 +40,19 @@ def main():
     parser.add_argument("--num-timesteps", type=int, help="Exact M1 or M3 environment step budget")
     parser.add_argument("--smoke", action="store_true", help="Run two PPO optimizer updates (pilot default: ten)")
     parser.add_argument("--diagnose-ppo", action="store_true",
-                        help="Record M1 KL and action saturation at each PPO update")
-    parser.add_argument("--actor-velocity", action="store_true",
-                        help="Append pelvis-frame forward and lateral velocity to the actor observation")
+                        help="Record M1 or M3 KL and action saturation at each PPO update")
+    actor = parser.add_mutually_exclusive_group()
+    actor.add_argument("--actor-velocity", action="store_true",
+                       help="Append pelvis-frame forward and lateral velocity to the actor observation")
+    actor.add_argument("--actor-upstream", action="store_true",
+                       help="Use the standard Playground G1 actor observation")
     parser.add_argument("--fixed-command-eval", action="store_true",
-                        help="Evaluate the final M1 checkpoint at 0 and 0.5 m/s forward")
+                        help="Evaluate the final M1 or M3 checkpoint at 0 and 0.5 m/s forward")
     args = parser.parse_args()
-    if args.diagnose_ppo and args.model != "M1":
-        parser.error("--diagnose-ppo applies to M1")
-    if args.actor_velocity and args.model not in ("M1", "M3"):
-        parser.error("--actor-velocity applies to M1 and M3")
-    if args.fixed_command_eval and args.model != "M1":
-        parser.error("--fixed-command-eval applies to M1")
+    if args.model not in ("M1", "M3"):
+        for option in ("diagnose_ppo", "actor_velocity", "actor_upstream", "fixed_command_eval"):
+            if getattr(args, option):
+                parser.error(f"--{option.replace('_', '-')} applies to M1 and M3")
     if args.history_len is None:
         args.history_len = 0 if args.model == "M1" else 5
     if args.model in ("M1", "M3"):
diff --git a/src/training/m3_env.py b/src/training/m3_env.py
index 4c44505..0d11ebf 100644
--- a/src/training/m3_env.py
+++ b/src/training/m3_env.py
@@ -9,7 +9,7 @@ from src.training.m3_history import initial_input, next_input
 
 
 class HistoryJoystick(joystick.Joystick):
-    def __init__(self, scene, config, length, mesh_directory, *, actor_velocity=False):
+    def __init__(self, scene, config, length, mesh_directory, *, actor_velocity=False, actor_upstream=False):
         mjx_env.MjxEnv.__init__(self, config)
         self._xml_path = str(scene)
         self._model_assets = base.get_assets()
@@ -24,8 +24,11 @@ class HistoryJoystick(joystick.Joystick):
         self._post_init()
         self.history_length = length
         self.actor_velocity = actor_velocity
+        self.actor_upstream = actor_upstream
 
     def _actor_obs(self, state):
+        if self.actor_upstream:
+            return state.obs["state"]
         data = state.data
         fields = (
             self.get_gyro(data, "pelvis"),
@@ -38,6 +41,12 @@ class HistoryJoystick(joystick.Joystick):
             fields += (self.get_local_linvel(data, "pelvis")[:2],)
         return jp.concatenate(fields)
 
+    def with_command(self, state, command):
+        """Set the command in the simulator and the current actor frame."""
+        start = 9 if self.actor_upstream else 6
+        actor = state.obs["state"].at[..., start:start + 3].set(command)
+        return state.replace(obs=dict(state.obs, state=actor), info=dict(state.info, command=command))
+
     def reset(self, rng):
         state = super().reset(rng)
         vector = initial_input(self._actor_obs(state), self.action_size, self.history_length, jp)
diff --git a/src/training/m3_pilot.py b/src/training/m3_pilot.py
index 74d6a81..4d6882b 100644
--- a/src/training/m3_pilot.py
+++ b/src/training/m3_pilot.py
@@ -71,10 +71,7 @@ def evaluate(env, policy, horizon, *, forward_command=None):
         command = jp.array([forward_command, 0., 0.], dtype=jp.float32)
         contact_sensor = env.mj_model.sensor_adr[env._right_foot_left_foot_found_sensor]
 
-        @jax.jit
-        def with_command(state):
-            state = state.replace(info=dict(state.info, command=command))
-            return state.replace(obs=dict(state.obs, state=env._actor_obs(state)))
+        with_command = jax.jit(partial(env.with_command, command=command))
 
         @jax.jit
         def forward_velocity(data):
@@ -210,12 +207,11 @@ def run(args):
     template = g1_constants.task_to_xml("flat_terrain")
     scene = write_scene(str(template), arena, out / "terrain")
     env = HistoryJoystick(scene, env_cfg, args.history_len, Path("results/m3_robot_assets"),
-                          actor_velocity=args.actor_velocity)
+                          actor_velocity=args.actor_velocity, actor_upstream=args.actor_upstream)
     observation_shapes = env.observation_size
-    actor_shape = jax.eval_shape(lambda key: env._actor_obs(env.reset(key)), jax.random.PRNGKey(0))
-    width = actor_shape.shape[-1]
-    expected = width + args.history_len * (width + env.action_size)
-    if observation_shapes["state"] != (expected,):
+    (actor_width,) = observation_shapes["state"]
+    width, remainder = divmod(actor_width - args.history_len * env.action_size, args.history_len + 1)
+    if width <= 0 or remainder:
         raise ValueError("Simulator observation shape disagrees with the configured actor width")
 
     ppo = locomotion_params.brax_ppo_config(TASK).to_dict()
@@ -236,7 +232,7 @@ def run(args):
                                                  else "PPO MLP with observation–action history stacking"),
         task=TASK, seed=args.seed, actor_fields=[
         "pelvis gyro", "projected gravity", "velocity command", "joint offsets", "joint velocities"],
-        observation_width=width, actions=env.action_size, actor_width=expected,
+        observation_width=width, actions=env.action_size, actor_width=actor_width,
         history_length=args.history_len, terrain_sha256=sha256(contents).hexdigest(),
         terrain=category, difficulty=difficulty, ppo=ppo, network=network,
         warp_contact_capacity=env_cfg.naconmax,
@@ -253,6 +249,13 @@ def run(args):
     if args.actor_velocity:
         settings["actor_velocity"] = True
         settings["actor_fields"].append("pelvis forward and lateral velocity")
+    if args.actor_upstream:
+        settings["actor_upstream"] = True
+        settings["actor_fields"] = [
+            "noisy pelvis linear velocity", "noisy pelvis gyro", "noisy projected gravity",
+            "velocity command", "noisy joint offsets", "noisy joint velocities",
+            "previous action", "gait phase cosine and sine",
+        ]
     print(f"Resolved {args.model} run configuration", flush=True)
     print(json.dumps(settings, indent=2, allow_nan=False), flush=True)
     save_json(out / "settings.json", settings)
'''
        applied = subprocess.run(["git", "apply", "--check", "-"], cwd=repo,
                                 input=upstream_patch, text=True, capture_output=True)
        if applied.returncode:
            raise RuntimeError("Use an updated m3-kaggle checkout with upstream actor support. " + applied.stderr)
        subprocess.run(["git", "apply", "-"], cwd=repo, input=upstream_patch, text=True, check=True)
    print("M1 upstream observation experiment ready.")

## Set up the training environment

The cell reuses a working environment or installs the training dependencies in a temporary environment for this Kaggle session. Enable Internet for the first install. For the observation comparisons, use the same runtime as the control. Set `m3_python` to that executable if needed.

In [ ]:
m3_python = None  # Set to the full M3 Python path if known.
venv_dir = Path(gettempdir()) / "bipedal-m1-training"
venv_python = str(venv_dir / "bin/python")
mode = "train" if needs_training else "verify"
expected_packages = {}
if needs_training and experiment == "upstream" and (run / "training.json").is_file():
    expected_packages = read(run, "settings.json")["packages"]

probe_code = """from importlib.metadata import version
import json
import sys
from src.training.m3_pilot import run
import jax

if sys.argv[1] == "train":
    devices = jax.local_devices()
    if len(devices) != 2 or any(
        d.platform not in ("gpu", "cuda") or "T4" not in d.device_kind.split()
        for d in devices
    ):
        raise SystemExit(f"M1 training requires Kaggle GPU T4 x2. JAX reported {devices}.")
    print("JAX devices:", ", ".join(f"{d} ({d.device_kind})" for d in devices))
versions = {name: version(name) for name in ("jax", "brax", "mujoco", "playground")}
for name, expected in json.loads(sys.argv[2]).items():
    if versions[name] != expected:
        raise SystemExit(f"The saved M1 control used {name} {expected}. This runtime has {versions[name]}.")
print("Versions:", ", ".join(f"{name} {number}" for name, number in versions.items()))
"""


def probe_runtime(python):
    return subprocess.run([python, "-c", probe_code, mode, json.dumps(expected_packages)], cwd=repo, env=env,
                          text=True, capture_output=True)


candidates = [m3_python] if m3_python is not None else [
    sys.executable, shutil.which("python"), shutil.which("python3"),
    "/opt/conda/bin/python", "/usr/local/bin/python3", repo / ".venv/bin/python", venv_python,
]
if m3_python is None:
    for variable in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        if env.get(variable):
            candidates.insert(0, Path(env[variable]) / "bin/python")

failures = []
for python in dict.fromkeys(str(path) for path in candidates if path):
    if not os.access(python, os.X_OK) or not Path(python).is_file():
        continue
    result = probe_runtime(python)
    if result.returncode == 0:
        training_python = python
        break
    failures.append(f"{python}: {result.stderr.strip() or result.stdout.strip()}")
else:
    if m3_python is not None:
        detail = failures[0] if failures else "Python executable unavailable."
        raise RuntimeError(f"Could not use m3_python={m3_python!r}. {detail}")

    if failures:
        print(failures[0], flush=True)
    print(f"Installing the M1 training stack in {venv_dir}", flush=True)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv_dir)], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "--python", venv_python, "install", "--no-cache-dir",
        "-r", str(repo / "requirements.txt"),
        "jax[cuda12]==0.11.1", "brax==0.14.2", "mujoco==4.0.1", "mujoco-mjx[warp]==4.0.1",
    ], cwd=repo, env=env, check=True)
    result = probe_runtime(venv_python)
    if result.returncode:
        raise RuntimeError(f"M1 runtime check failed in {venv_python}. {result.stderr or result.stdout}")
    training_python = venv_python

print(f"M1 training Python: {training_python}")
print(result.stdout.strip())

## Train and verify M1

The notebook verifies each writable completed run in a fresh Python process. It reads the existing verification from a saved control attached as an input. For a fresh start after an interrupted run, choose a new output directory in the first code cell.

In [ ]:
def train_or_verify_m1(folder, *, actor="control", attached=False):
    width = {"control": 67, "velocity": 69, "upstream": 103}[actor]

    def check_run():
        settings = read(folder, "settings.json")
        ppo = settings["ppo"]
        expected = ("M1", 0, seed, num_envs, budget, 0.0001, width, width)
        actual = (settings["model"], settings["history_length"], settings["seed"],
                  ppo["num_envs"], ppo["num_timesteps"], ppo["learning_rate"],
                  settings["actor_width"], settings["observation_width"])
        if actual != expected or read(folder, "training.json")["environment_steps"] != budget:
            raise ValueError(f"{folder} does not contain the requested completed M1 run.")
        if settings["network"]["policy_obs_key"] != "state":
            raise ValueError("Expected an M1 actor using only the current state.")
        if actor == "upstream" and settings.get("actor_upstream") is not True:
            raise ValueError(f"Expected the standard Playground G1 observation in {folder}.")
        if variant_run is not None and not (folder / "fixed_command_evaluation.json").is_file():
            raise FileNotFoundError(f"Fixed-command evaluation is missing in {folder}. Use a fresh run folder.")
        return settings

    if (folder / "training.json").is_file():
        settings = check_run()
        if not attached:
            command = [training_python, "-u", "-m", "src.training.m3_pilot", str(folder)]
            subprocess.run(command, cwd=repo, env=env, check=True)
    else:
        if folder.exists() and any(folder.iterdir()):
            raise FileExistsError(f"Partial run at {folder}. Choose an empty output directory.")
        print(f"Training the {width}-input M1 with {num_envs // 2} environments per GPU.", flush=True)
        command = [training_python, "-u", "scripts/train_kaggle.py", "--model", "M1",
                   "--history-len", "0", "--num-envs", str(num_envs), "--num-timesteps", str(budget),
                   "--seed", str(seed), "--output-dir", str(folder)]
        if variant_run is not None:
            command += ["--diagnose-ppo", "--fixed-command-eval"]
        if actor != "control":
            command.append(f"--actor-{actor}")
        subprocess.run(command, cwd=repo, env=env, check=True)
        settings = check_run()
    reload_result = read(folder, "reload.json")
    if reload_result["action_verified"] is not True or int(reload_result["checkpoint"]) != budget:
        raise ValueError(f"Expected a verified M1 checkpoint at step {budget}.")
    print(f"Verified {width}-input M1 checkpoint at {budget:,} steps in {folder}")
    return settings


m1 = train_or_verify_m1(run, attached=control_reference is not None)
if variant_run is not None:
    m1_variant = train_or_verify_m1(variant_run, actor=experiment)

## Evaluate the M1 runs

The joystick results use the same ten seeds. The observation experiment table compares actual forward speed, episode duration, and foot collisions for the same stand-still and 0.50 m/s commands.

In [ ]:
def report(folder, settings, expected_seeds=None):
    print("Policy / stage       Duration    Tracking error    Return    Early ends")
    for stage in ("initial", "trained"):
        episodes = read(folder, f"{stage}_evaluation.json")["episodes"]
        seeds = sorted(e["seed"] for e in episodes)
        if len(seeds) != 10 or len(set(seeds)) != 10:
            raise ValueError(f"Expected ten distinct {stage} evaluation seeds in {folder}.")
        if expected_seeds is not None and seeds != expected_seeds:
            raise ValueError(f"Unmatched {stage} evaluation seeds in {folder}.")
        expected_seeds = seeds
        duration = statistics.mean(e["steps"] for e in episodes) * settings["control_dt"]
        error = statistics.mean(e["tracking_error_mps"] for e in episodes)
        reward = statistics.mean(e["episode_return"] for e in episodes)
        ended = sum(e["fall"] for e in episodes)
        print(f'{settings["model"]} / {stage:<8}      {duration:.3f} s       {error:.3f} m/s'
              f'     {reward:7.3f}      {ended}/10')
    return expected_seeds

m1_seeds = report(run, m1)
if variant_run is not None:
    report(variant_run, m1_variant, m1_seeds)
    for key in ("ppo", "network", "packages", "devices", "terrain_sha256", "terrain", "difficulty",
                "control_dt", "physics_dt", "warp_contact_capacity", "simulator_backend", "task", "actions"):
        if m1[key] != m1_variant[key]:
            raise ValueError(f"Run settings differ for {key}. Use the same training configuration and runtime as the control.")

    critic_key = m1["network"]["value_obs_key"]
    if m1["observation_shapes"][critic_key] != m1_variant["observation_shapes"][critic_key]:
        raise ValueError("Critic input shapes differ.")

    print("Actor    Target    Actual vx    Duration    Tracking error    Foot collisions    Finished")
    for folder, settings in ((run, m1), (variant_run, m1_variant)):
        scenarios = read(folder, "fixed_command_evaluation.json")["scenarios"]
        if [scenario["forward_command_mps"] for scenario in scenarios] != [0.0, 0.5]:
            raise ValueError(f"Expected stand-still and straight-forward evaluations in {folder}.")
        for scenario in scenarios:
            episodes = scenario["episodes"]
            if sorted(e["seed"] for e in episodes) != m1_seeds:
                raise ValueError(f"Unmatched fixed-command evaluation seeds in {folder}.")
            duration = statistics.mean(e["steps"] for e in episodes) * settings["control_dt"]
            forward = statistics.mean(e["mean_forward_mps"] for e in episodes)
            error = statistics.mean(e["tracking_error_mps"] for e in episodes)
            contacts = sum(e["foot_to_foot_contact"] for e in episodes)
            finished = sum(not e["fall"] for e in episodes)
            print(f'{settings["actor_width"]:>3}    {scenario["forward_command_mps"]:>5.2f} m/s'
                  f'    {forward:>6.3f} m/s    {duration:>6.3f} s    {error:>6.3f} m/s'
                  f'         {contacts:>2}/10         {finished:>2}/10')

## Optional comparison with M3

In baseline mode, attach a saved M3 run and rerun the next cell. Set `reference` to the saved run directory if it has a different name or there are multiple matches.

In [ ]:
if variant_run is None:
    reference = None  # Optionally set to Path("/kaggle/input/.../your-m3-run").
    if reference is None:
        name = "m3_1p6m_env8_lr1e4"
        local = repo / "results" / name
        if (local / "settings.json").is_file():
            reference = local
        else:
            matches = sorted(inputs.rglob(f"{name}/settings.json"))
            if len(matches) == 1:
                reference = matches[0].parent
            elif len(matches) > 1:
                print("Multiple M3 runs found. Set reference to one of:", [p.parent for p in matches])

    if reference is None:
        print("M1 evaluation complete. M3 comparison skipped.")
    else:
        m3 = read(reference, "settings.json")
        if (m3["model"], m3["history_length"]) != ("M3", 5):
            raise ValueError("Select a five-step M3 reference run.")
        if read(reference, "training.json")["environment_steps"] != budget:
            raise ValueError(f"Expected a completed {budget:,}-step M3 run.")
        for key in ("task", "seed", "terrain_sha256", "terrain", "difficulty", "ppo", "network",
                    "actor_fields", "observation_width", "actions", "control_dt", "physics_dt",
                    "simulator_backend", "warp_contact_capacity"):
            if m1[key] != m3[key]:
                raise ValueError(f"Run settings differ for {key}: {m1[key]} vs {m3[key]}")
        critic_key = m1["network"]["value_obs_key"]
        if m1["observation_shapes"][critic_key] != m3["observation_shapes"][critic_key]:
            raise ValueError("Critic input shapes differ.")
        if m1["packages"] != m3["packages"]:
            print("Runtime versions differed between runs. M1:", m1["packages"], "M3:", m3["packages"])
        report(reference, m3, m1_seeds)